# Notebook 2S — SoilGrids visual analytics and ML-table construction

This notebook builds the **SoilGrids-centered modelling table** and manuscript-style exploratory figures. It creates one row per **taxon × occurrence/spatial block × compound**, plus an aggregated primary **taxon × compound** table for Notebook 3S.

In [ ]:
import os, re, io, json, time, math, hashlib, pathlib, warnings, datetime as dt
from typing import Dict, List, Optional, Tuple
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 220)
pd.set_option('display.width', 180)

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
try:
    from scipy.cluster.hierarchy import linkage, leaves_list
    from scipy.stats import spearmanr
    SCIPY_AVAILABLE=True
except Exception:
    SCIPY_AVAILABLE=False

In [ ]:
CONFIG={'project_dir':'soilgrids_notebook2_outputs','input_dirs':['.','/content','/mnt/data','/content/soilgrids_medicinal_value_dataset','/content/soilgrids_medicinal_value_dataset/derived','/content/soilgrids_medicinal_value_dataset/soilgrids','/content/soilgrids_medicinal_value_dataset/metadata'],'curated_positive_pairs':[('TAX_RHODIOLA','CMP_SALIDROSIDE'),('TAX_RHODIOLA','CMP_TYROSOL'),('TAX_HYPERICUM','CMP_HYPERICIN'),('TAX_HYPERICUM','CMP_TOTAL_FLAVONOIDS'),('TAX_HYPERICUM','CMP_TOTAL_PHENOLICS'),('TAX_ARTEMISIA_ANNUA','CMP_ARTEMISININ'),('TAX_GLYCYRRHIZA_GLABRA','CMP_GLYCYRRHIZIN'),('TAX_SEDUM','CMP_TOTAL_FLAVONOIDS'),('TAX_SEDUM','CMP_TOTAL_PHENOLICS'),('TAX_TAXUS','CMP_PACLITAXEL')],'figure_dpi':300,'save_pdf':True,'random_seed':42}
PROJECT=pathlib.Path(CONFIG['project_dir']).resolve(); DIRS={'derived':PROJECT/'derived','figures':PROJECT/'figures','tables':PROJECT/'tables'}
for d in DIRS.values(): d.mkdir(parents=True, exist_ok=True)
RUN_ID=dt.datetime.now(dt.timezone.utc).strftime('%Y%m%dT%H%M%SZ')

In [ ]:
def utc_now():
    return dt.datetime.now(dt.timezone.utc).isoformat(timespec='seconds')

def safe_name(x, max_len=160):
    return re.sub(r'[^A-Za-z0-9._-]+','_',str(x)).strip('_')[:max_len]

def write_table(df, path_stem):
    path_stem = pathlib.Path(path_stem)
    path_stem.parent.mkdir(parents=True, exist_ok=True)
    tsv = path_stem.with_suffix('.tsv')
    df.to_csv(tsv, sep='\t', index=False)
    try:
        df.to_parquet(path_stem.with_suffix('.parquet'), index=False)
    except Exception as e:
        print('Parquet skipped:', e)
    return tsv

def sha256_file(path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path,'rb') as f:
        while True:
            b = f.read(chunk_size)
            if not b: break
            h.update(b)
    return h.hexdigest()

def request_json(url, params=None, session=None, retries=5, sleep=0.25, timeout=120):
    session = session or requests.Session()
    last = None
    for i in range(retries):
        try:
            r = session.get(url, params=params, timeout=timeout)
            if r.status_code in (429,500,502,503,504):
                last = f'HTTP {r.status_code}: {r.text[:250]}'
                time.sleep(sleep*(2**i)); continue
            r.raise_for_status()
            time.sleep(sleep)
            return r.json()
        except Exception as e:
            last = str(e)
            time.sleep(sleep*(2**i))
    raise RuntimeError(f'GET failed: {url}; params={params}; last={last}')

def save_figure(fig, name, fig_dir, dpi=300, save_pdf=True):
    fig_dir = pathlib.Path(fig_dir); fig_dir.mkdir(parents=True, exist_ok=True)
    png = fig_dir / f'{safe_name(name)}.png'
    fig.savefig(png, dpi=dpi, bbox_inches='tight')
    if save_pdf:
        fig.savefig(fig_dir / f'{safe_name(name)}.pdf', bbox_inches='tight')
    print('Saved:', png)
    return png

def add_panel_label(ax, label):
    ax.text(-0.10, 1.08, label, transform=ax.transAxes, fontsize=15, fontweight='bold', va='top')

def find_input_file(filename, required=True):
    candidates=[]
    for d in CONFIG['input_dirs']:
        d=pathlib.Path(d); candidates.append(d/filename)
        if d.exists() and d.is_dir(): candidates.extend(d.rglob(filename))
    existing=sorted(set([p for p in candidates if p.exists()]), key=lambda p:(len(str(p)),str(p)))
    if not existing:
        if required: raise FileNotFoundError(filename)
        return None
    return existing[0]
def load_tsv(filename, required=True):
    p=find_input_file(filename, required)
    if p is None: return pd.DataFrame()
    df=pd.read_csv(p, sep='\t', dtype=str, low_memory=False); print('Loaded', filename, df.shape, p); return df
def spearman_safe(x,y):
    if not SCIPY_AVAILABLE: return np.nan
    x=pd.to_numeric(pd.Series(x),errors='coerce'); y=pd.to_numeric(pd.Series(y),errors='coerce'); ok=x.notna()&y.notna()
    if ok.sum()<3 or x[ok].nunique()<=1 or y[ok].nunique()<=1: return np.nan
    return spearmanr(x[ok],y[ok]).correlation

In [ ]:
occ_soil_df=load_tsv('occurrence_soilgrids_features.tsv')
soil_long_df=load_tsv('soilgrids_point_features_long.tsv')
target_taxa_df=load_tsv('target_taxa_config.tsv')
target_compounds_df=load_tsv('target_compounds_config.tsv')
soil_feature_cols=[c for c in occ_soil_df.columns if c.startswith('soil_')]
print('Soil features:', len(soil_feature_cols))

In [ ]:
# Target evidence: curated positive pairs; other pairs are pseudo-negative/background.
curated=set(CONFIG['curated_positive_pairs'])
rows=[]
for _,tax in target_taxa_df.iterrows():
    for _,comp in target_compounds_df.iterrows():
        pair=(tax['taxon_id'], comp['compound_id']); pos=pair in curated
        rows.append({'taxon_id':tax['taxon_id'],'scientific_name':tax['scientific_name'],'family':tax.get('family',''),'rank':tax.get('rank',''),'compound_id':comp['compound_id'],'compound_name':comp['compound_name'],'compound_class':comp.get('compound_class',''),'compound_record_type':comp.get('record_type',''),'target_evidence_score':3 if pos else 0,'target_label_for_open_data_classifier':int(pos),'target_evidence_class':'curated_strong_positive' if pos else 'background_pseudo_negative','target_warning':'Score 0 is background/pseudo-negative, not confirmed absence.'})
target_df=pd.DataFrame(rows)
write_table(target_df, DIRS['derived']/'soilgrids_taxon_compound_target_table')
ml_table=occ_soil_df.merge(target_df.drop(columns=['scientific_name','family','rank'],errors='ignore'),on='taxon_id',how='inner')
ml_table['taxonomic_group_id']=ml_table.get('family','').astype(str)+'__'+ml_table['taxon_id'].astype(str)
ml_table['compound_group_id']=ml_table['compound_class'].astype(str)
ml_table['spatial_group_id']=ml_table['spatial_block_id'].astype(str)
ml_table['source_group_id']='soilgrids_gbif_open_data'
ml_table=ml_table.reset_index(drop=True); ml_table['ml_row_id']=['SOILML_'+str(i).zfill(9) for i in range(len(ml_table))]
write_table(ml_table, DIRS['derived']/'soilgrids_taxon_occurrence_compound_ml_table')
named={'target_evidence_score':('target_evidence_score','max'),'target_label_for_open_data_classifier':('target_label_for_open_data_classifier','max'),'n_occurrences':('ml_row_id','count'),'n_spatial_blocks':('spatial_group_id','nunique')}
for c in soil_feature_cols:
    named[c+'__mean']=(c,'mean'); named[c+'__sd']=(c,'std')
primary_df=ml_table.groupby(['taxon_id','input_taxon','compound_id','compound_name','compound_class'],dropna=False).agg(**named).reset_index().rename(columns={'input_taxon':'scientific_name'})
write_table(primary_df, DIRS['derived']/'soilgrids_primary_taxon_compound_table')
print(ml_table.shape, primary_df.shape)

In [ ]:
# Figure 1: design and coverage
fig,axes=plt.subplots(2,2,figsize=(15,10)); axes=axes.reshape(-1)
counts=occ_soil_df.groupby('input_taxon').size().sort_values(); axes[0].barh(counts.index, counts.values); axes[0].set_title('SoilGrids-linked occurrence coverage'); axes[0].set_xlabel('Occurrences'); add_panel_label(axes[0],'A')
blocks=occ_soil_df.groupby('input_taxon')['spatial_block_id'].nunique().sort_values(); axes[1].barh(blocks.index, blocks.values); axes[1].set_title('Spatial-block representation'); axes[1].set_xlabel('Spatial blocks'); add_panel_label(axes[1],'B')
for taxon,sub in occ_soil_df.groupby('input_taxon'):
    axes[2].scatter(pd.to_numeric(sub['decimal_longitude'],errors='coerce'),pd.to_numeric(sub['decimal_latitude'],errors='coerce'),s=6,alpha=.35,label=taxon)
axes[2].set_xlim(-180,180); axes[2].set_ylim(-90,90); axes[2].grid(True,alpha=.3); axes[2].set_title('Occurrence space for SoilGrids extraction'); axes[2].set_xlabel('Longitude'); axes[2].set_ylabel('Latitude'); axes[2].legend(fontsize=7,frameon=False,bbox_to_anchor=(1.02,.5),loc='center left'); add_panel_label(axes[2],'C')
tc=primary_df['target_evidence_score'].value_counts().sort_index(); axes[3].bar([str(x) for x in tc.index],tc.values); axes[3].set_title('Target evidence structure'); axes[3].set_xlabel('Target evidence score'); axes[3].set_ylabel('Rows'); add_panel_label(axes[3],'D')
fig.suptitle('SoilGrids medicinal-plant dataset design',fontsize=16,y=1.02); fig.tight_layout(); save_figure(fig,'figure1_soilgrids_dataset_design',DIRS['figures'],CONFIG['figure_dpi'],CONFIG['save_pdf']); plt.show()

In [ ]:
# Depth profiles
if not soil_long_df.empty:
    plot_props=['phh2o','soc','clay','sand','cec','nitrogen']; depth_order=['0-5cm','5-15cm','15-30cm','30-60cm','60-100cm','100-200cm']
    long=soil_long_df.copy(); long['scaled_value']=pd.to_numeric(long['scaled_value'],errors='coerce'); long=long[(long['soil_property'].isin(plot_props))&(long['value_type'].eq('mean'))]; long['depth_label']=pd.Categorical(long['depth_label'],categories=depth_order,ordered=True)
    fig,axes=plt.subplots(2,3,figsize=(15,9)); axes=axes.reshape(-1)
    for ax,prop in zip(axes,plot_props):
        prof=long[long['soil_property'].eq(prop)].groupby('depth_label')['scaled_value'].agg(['mean','std']).reindex(depth_order)
        ax.errorbar(prof['mean'],range(len(prof)),xerr=prof['std'],marker='o'); ax.set_yticks(range(len(depth_order))); ax.set_yticklabels(depth_order); ax.invert_yaxis(); ax.set_title(prop); ax.set_xlabel('Mean scaled value')
    fig.suptitle('SoilGrids depth profiles across query points',fontsize=16,y=1.02); fig.tight_layout(); save_figure(fig,'figure2_soilgrids_depth_profiles',DIRS['figures'],CONFIG['figure_dpi'],CONFIG['save_pdf']); plt.show()

In [ ]:
# Correlation and PCA
soil_cols=[c for c in primary_df.columns if c.startswith('soil_')]
soil_numeric=primary_df[soil_cols].apply(pd.to_numeric,errors='coerce'); soil_numeric=soil_numeric.loc[:,soil_numeric.notna().sum()>2]; soil_numeric=soil_numeric.loc[:,soil_numeric.nunique(dropna=True)>1]
corr=soil_numeric.corr(method='spearman').fillna(0)
if SCIPY_AVAILABLE and corr.shape[0]>2:
    dist=1-np.abs(corr.values); np.fill_diagonal(dist,0); order=leaves_list(linkage(dist,method='average'))
else: order=np.arange(corr.shape[0])
corr_ordered=corr.iloc[order,order]
write_table(corr.reset_index().rename(columns={'index':'feature'}), DIRS['derived']/'soil_feature_spearman_correlation')
X=SimpleImputer(strategy='median').fit_transform(soil_numeric); X=StandardScaler().fit_transform(X)
pca=PCA(n_components=min(5,X.shape[0],X.shape[1]),random_state=CONFIG['random_seed']); scores=pca.fit_transform(X)
pca_scores=primary_df[['taxon_id','scientific_name','compound_id','compound_name','target_evidence_score']].copy()
for i in range(scores.shape[1]): pca_scores[f'PC{i+1}']=scores[:,i]
pca_load=pd.DataFrame(pca.components_.T,index=soil_numeric.columns,columns=[f'PC{i+1}' for i in range(scores.shape[1])]).reset_index().rename(columns={'index':'feature'})
write_table(pca_scores,DIRS['derived']/'soil_pca_scores_taxon_compound'); write_table(pca_load,DIRS['derived']/'soil_pca_loadings')
# Figure
fig,axes=plt.subplots(2,2,figsize=(16,12)); axes=axes.reshape(-1)
im=axes[0].imshow(corr_ordered.values,vmin=-1,vmax=1); axes[0].set_title('Clustered SoilGrids feature correlation'); fig.colorbar(im,ax=axes[0],shrink=.7); add_panel_label(axes[0],'A')
sc=axes[1].scatter(pca_scores['PC1'],pca_scores['PC2'],c=pca_scores['target_evidence_score'],s=70,alpha=.8); axes[1].set_title('PCA of SoilGrids feature space'); axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)'); axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)'); fig.colorbar(sc,ax=axes[1],shrink=.7); add_panel_label(axes[1],'B')
load=pca_load.copy(); load['strength']=np.sqrt(load['PC1']**2+load['PC2']**2); top=load.sort_values('strength',ascending=False).head(15).iloc[::-1]; axes[2].barh(top['feature'],top['strength']); axes[2].set_title('Dominant soil PCA features'); add_panel_label(axes[2],'C')
tc=[{'feature':c,'spearman_with_target':spearman_safe(primary_df[c],primary_df['target_evidence_score'])} for c in soil_numeric.columns]; tcd=pd.DataFrame(tc).dropna().sort_values('spearman_with_target',key=lambda s:s.abs(),ascending=False); write_table(tcd,DIRS['derived']/'soil_feature_target_spearman'); tt=tcd.head(15).iloc[::-1]; axes[3].barh(tt['feature'],tt['spearman_with_target']); axes[3].axvline(0,lw=.8); axes[3].set_title('Soil feature-target associations'); add_panel_label(axes[3],'D')
fig.suptitle('SoilGrids feature relationships and PCA',fontsize=16,y=1.02); fig.tight_layout(); save_figure(fig,'figure3_soilgrids_correlation_pca_relationships',DIRS['figures'],CONFIG['figure_dpi'],CONFIG['save_pdf']); plt.show()

In [ ]:
# Handoff
validation=ml_table[['ml_row_id','taxon_id','compound_id','spatial_group_id','taxonomic_group_id','compound_group_id','source_group_id','target_evidence_score','target_label_for_open_data_classifier']].copy()
validation['recommended_primary_cv']='taxon_and_compound_grouped'; validation['spatial_cv_note']='Occurrence-level spatial holdout is sensitivity only.'
write_table(validation,DIRS['derived']/'soilgrids_validation_group_table_for_notebook3')
write_table(ml_table,DIRS['derived']/'soilgrids_occurrence_level_spatial_sensitivity_table')
handoff=pd.DataFrame([{'file_role':'primary_ml_table','path':str(DIRS['derived']/'soilgrids_primary_taxon_compound_table.tsv'),'required_for_notebook3':True},{'file_role':'occurrence_spatial_sensitivity_table','path':str(DIRS['derived']/'soilgrids_occurrence_level_spatial_sensitivity_table.tsv'),'required_for_notebook3':True},{'file_role':'validation_groups','path':str(DIRS['derived']/'soilgrids_validation_group_table_for_notebook3.tsv'),'required_for_notebook3':True}])
handoff['exists_and_nonempty']=handoff['path'].apply(lambda x:pathlib.Path(x).exists() and pathlib.Path(x).stat().st_size>0)
write_table(handoff,DIRS['derived']/'soilgrids_notebook2_handoff_manifest'); display(handoff)